<a href="https://colab.research.google.com/github/md-shafiq-cloud-architect/aws-cloud-security-portfolio/blob/main/03-iam-identity-center-genai-security/iam_identity_center_genai_security_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q boto3

import boto3
from botocore.exceptions import NoCredentialsError

try:
    sso_client = boto3.client('sso-admin', region_name='us-east-1')
    response = sso_client.list_instances()
    instances = response.get('Instances', [])

    if instances:
        print("IAM Identity Center is active.")
        for inst in instances:
            print(f"Instance ARN: {inst['InstanceArn']}")
            print(f"Identity Store ID: {inst['IdentityStoreId']}")
    else:
        print("No active IAM Identity Center instance found.")
except NoCredentialsError:
    print("boto3 installed successfully! Next step: Configure AWS credentials.")
except Exception as e:
    print(f"Error: {e}")

boto3 installed successfully! Next step: Configure AWS credentials.


In [3]:
# Validate Identity Provider compatibility with Trusted Token Issuer for Amazon Q

UNSUPPORTED_IDPS = ["Active Directory Domain Service"]
SUPPORTED_IDPS = [
    "Azure Entra ID",
    "Google Workspace",
    "Okta",
    "Ping",
    "OneLogin",
    "CyberArk",
    "JumpCloud",
    "IAM Identity Center Directory"
]

def validate_trusted_token_issuer(idp_name):
    print(f"Checking Identity Provider: {idp_name}")
    if idp_name in UNSUPPORTED_IDPS:
        print("STATUS: BLOCKED - Trusted Token Issuers do not support Active Directory Domain Service.")
    elif idp_name in SUPPORTED_IDPS:
        print("STATUS: APPROVED - Compatible with Trusted Token Issuer for Amazon Q Business.")
    else:
        print("STATUS: WARNING - Verify OIDC/SAML 2.0 federation compatibility.")

# Test cases based on architecture rules
validate_trusted_token_issuer("Okta")
print("-" * 40)
validate_trusted_token_issuer("Active Directory Domain Service")

Checking Identity Provider: Okta
STATUS: APPROVED - Compatible with Trusted Token Issuer for Amazon Q Business.
----------------------------------------
Checking Identity Provider: Active Directory Domain Service
STATUS: BLOCKED - Trusted Token Issuers do not support Active Directory Domain Service.


In [4]:
# Pre-deployment security check for CloudFormation stack setup

REQUIRED_REGION = "us-east-1"
REQUIRED_CAPABILITY = "CAPABILITY_NAMED_IAM"

def validate_stack_deployment(target_region, capabilities, is_sandbox):
    print(f"Validating deployment in region: {target_region}")

    if not is_sandbox:
        print("SECURITY ALERT: Do not deploy lab resources in a production account!")
        return
    if target_region != REQUIRED_REGION:
        print(f"REGION ERROR: Must deploy in locked region ({REQUIRED_REGION}).")
        return
    if REQUIRED_CAPABILITY not in capabilities:
        print(f"IAM ERROR: Missing {REQUIRED_CAPABILITY} acknowledgment.")
        return

    print("STATUS: READY - CloudFormation stack (ws-template.yaml) approved for launch.")

# Test valid sandbox deployment configuration
validate_stack_deployment(
    target_region="us-east-1",
    capabilities=["CAPABILITY_NAMED_IAM"],
    is_sandbox=True
)

Validating deployment in region: us-east-1
STATUS: READY - CloudFormation stack (ws-template.yaml) approved for launch.


In [5]:
# Audit deployed AWS resources and lab execution sequence

DEPLOYED_RESOURCES = {
    "Compute": "Amazon EC2 (Hosts VS Code Server & Custom Web App)",
    "CDN": "Amazon CloudFront (Fast delivery to VS Code Server)",
    "TokenIssuer": "Amazon Cognito User Pool (Optional default token issuer)",
    "Storage": "Amazon S3 Bucket (Stores logs and static assets)",
    "Automation": "AWS Lambda (Handles CloudFormation custom resources)"
}

LAB_WORKFLOW = [
    "Step 1: Enable AWS IAM Identity Center",
    "Step 2: Configure Identity Provider & Assign Users/Groups",
    "Step 3: Set up & test Amazon Q Developer in VS Code Server",
    "Step 4: Create Amazon Q Business App & verify Web Experience URL",
    "Step 5: Integrate Amazon Q Business into Custom Employee Portal"
]

def run_architecture_audit():
    print("--- DEPLOYED INFRASTRUCTURE INVENTORY ---")
    for role, service in DEPLOYED_RESOURCES.items():
        print(f"[{role}] -> {service}")

    print("\n--- EXECUTION WORKFLOW ---")
    for step in LAB_WORKFLOW:
        print(step)

run_architecture_audit()

--- DEPLOYED INFRASTRUCTURE INVENTORY ---
[Compute] -> Amazon EC2 (Hosts VS Code Server & Custom Web App)
[CDN] -> Amazon CloudFront (Fast delivery to VS Code Server)
[TokenIssuer] -> Amazon Cognito User Pool (Optional default token issuer)
[Storage] -> Amazon S3 Bucket (Stores logs and static assets)
[Automation] -> AWS Lambda (Handles CloudFormation custom resources)

--- EXECUTION WORKFLOW ---
Step 1: Enable AWS IAM Identity Center
Step 2: Configure Identity Provider & Assign Users/Groups
Step 3: Set up & test Amazon Q Developer in VS Code Server
Step 4: Create Amazon Q Business App & verify Web Experience URL
Step 5: Integrate Amazon Q Business into Custom Employee Portal


In [6]:
# Validate IAM Identity Center Lab Preparation & Security Posture

def validate_identity_center_setup(env_mode, instance_type, email_otp_enabled, mfa_mode):
    print(f"--- IAM IDENTITY CENTER AUDIT ({env_mode.upper()}) ---")

    # Check Instance Type rule
    if env_mode == "production" and instance_type != "Organization":
        print("CRITICAL: Production must use an Organization instance!")
    else:
        print(f"Instance Type ({instance_type}): OK")

    # Check API User OTP rule
    if email_otp_enabled:
        print("Email OTP for API Users: ENABLED (Ready for CreateUser API)")
    else:
        print("WARNING: Enable Email OTP so API-created users can sign in.")

    # Check MFA security trade-off
    if env_mode == "lab" and mfa_mode == "Never":
        print("MFA Policy: DISABLED (Acceptable for lab only; must enable in production)")
    elif env_mode == "production" and mfa_mode == "Never":
        print("SECURITY VIOLATION: MFA cannot be disabled in production!")

# Run audit for current Lab 1 configuration
validate_identity_center_setup(
    env_mode="lab",
    instance_type="Account",
    email_otp_enabled=True,
    mfa_mode="Never"
)

--- IAM IDENTITY CENTER AUDIT (LAB) ---
Instance Type (Account): OK
Email OTP for API Users: ENABLED (Ready for CreateUser API)
MFA Policy: DISABLED (Acceptable for lab only; must enable in production)


In [7]:
# Audit Identity Provider federation protocols and SCIM sync capabilities

IDP_CONFIGURATIONS = {
    "Identity Center Directory": {
        "protocol": "Native / API (create_idp_user.py)",
        "scim_user_sync": True,
        "scim_group_sync": True,
        "notes": "Default source. Requires Email OTP enabled for API-created users."
    },
    "Okta": {
        "protocol": "SAML 2.0 + SCIM v2.0",
        "scim_user_sync": True,
        "scim_group_sync": True,
        "notes": "Supports full automated user and group push (Qusers) via SCIM."
    },
    "Microsoft Entra ID": {
        "protocol": "SAML 2.0 + SCIM v2.0",
        "scim_user_sync": True,
        "scim_group_sync": False,
        "notes": "Free tier syncs users only; group assignment requires Entra ID P2 license."
    },
    "Google Workspace": {
        "protocol": "SAML 2.0 + SCIM v2.0",
        "scim_user_sync": True,
        "scim_group_sync": False,
        "notes": "Group sync limited; use 'aws identitystore create-group' CLI as fallback."
    }
}

def audit_idp_federation(idp_name):
    config = IDP_CONFIGURATIONS.get(idp_name)
    if not config:
        print(f"ERROR: Unknown Identity Provider '{idp_name}'")
        return

    print(f"=== IDP FEDERATION AUDIT: {idp_name} ===")
    print(f"Auth Protocol   : {config['protocol']}")
    print(f"Auto User Sync  : {config['scim_user_sync']}")
    print(f"Auto Group Sync : {config['scim_group_sync']}")
    print(f"Architect Note  : {config['notes']}\n")

# Run audit across all 4 Lab 2 providers
for provider in IDP_CONFIGURATIONS:
    audit_idp_federation(provider)

=== IDP FEDERATION AUDIT: Identity Center Directory ===
Auth Protocol   : Native / API (create_idp_user.py)
Auto User Sync  : True
Auto Group Sync : True
Architect Note  : Default source. Requires Email OTP enabled for API-created users.

=== IDP FEDERATION AUDIT: Okta ===
Auth Protocol   : SAML 2.0 + SCIM v2.0
Auto User Sync  : True
Auto Group Sync : True
Architect Note  : Supports full automated user and group push (Qusers) via SCIM.

=== IDP FEDERATION AUDIT: Microsoft Entra ID ===
Auth Protocol   : SAML 2.0 + SCIM v2.0
Auto User Sync  : True
Auto Group Sync : False
Architect Note  : Free tier syncs users only; group assignment requires Entra ID P2 license.

=== IDP FEDERATION AUDIT: Google Workspace ===
Auth Protocol   : SAML 2.0 + SCIM v2.0
Auto User Sync  : True
Auto Group Sync : False
Architect Note  : Group sync limited; use 'aws identitystore create-group' CLI as fallback.



In [8]:
# Audit Amazon Q Developer IDE access, subscription status, and session duration

def check_q_developer_access(user_email, profile_name, is_subscribed, auth_code_verified):
    print(f"--- AMAZON Q DEVELOPER ACCESS AUDIT: {user_email} ---")

    if not is_subscribed:
        print("ERROR: AccessDeniedException - User/Group not subscribed in Q Developer Console.")
        return

    if not auth_code_verified:
        print("PENDING: Complete 8-digit device authorization code check in browser.")
        return

    standard_sso_hours = 8
    q_dev_session_days = 90

    print(f"Profile Name     : {profile_name} (AWS Managed Application)")
    print(f"IDE Status       : AUTHORIZED (Amazon Q Chat Window Active)")
    print(f"Session Duration : {q_dev_session_days} days (vs {standard_sso_hours}h standard SSO session)")

# Test valid developer access flow
check_q_developer_access(
    user_email="developer@example.com",
    profile_name="QDevProfile-us-east-1-001",
    is_subscribed=True,
    auth_code_verified=True
)

--- AMAZON Q DEVELOPER ACCESS AUDIT: developer@example.com ---
Profile Name     : QDevProfile-us-east-1-001 (AWS Managed Application)
IDE Status       : AUTHORIZED (Amazon Q Chat Window Active)
Session Duration : 90 days (vs 8h standard SSO session)


In [9]:
# Validate Amazon Q Business app configuration and user access routing

def audit_q_business_app(app_name, idc_region, app_region, assigned_group, access_method):
    print(f"--- AMAZON Q BUSINESS AUDIT: {app_name} ---")

    # Check Cross-region call requirement
    cross_region_needed = (idc_region != app_region)
    print(f"Regions Match    : {not cross_region_needed} (Cross-region calls needed: {cross_region_needed})")
    print(f"Assigned Group   : {assigned_group} (Q Business Pro Subscription)")
    print(f"Registry Status  : Registered as AWS Managed Application")

    # Validate user access path
    if access_method == "deployed_url":
        print("Access Route     : Direct Deployed URL (https://*.on.aws) -> Redirects to IdP if no active session.")
    elif access_method == "access_portal":
        print("Access Route     : Centralized AWS Access Portal (https://*.awsapps.com) -> App Tile Launcher.")
    else:
        print("ERROR: Invalid access method specified.")

# Test Lab 4 configuration with AWS Access Portal
audit_q_business_app(
    app_name="MyQBusinessApp",
    idc_region="us-east-1",
    app_region="us-east-1",
    assigned_group="Qusers",
    access_method="access_portal"
)

--- AMAZON Q BUSINESS AUDIT: MyQBusinessApp ---
Regions Match    : True (Cross-region calls needed: False)
Assigned Group   : Qusers (Q Business Pro Subscription)
Registry Status  : Registered as AWS Managed Application
Access Route     : Centralized AWS Access Portal (https://*.awsapps.com) -> App Tile Launcher.


In [10]:
# Simulate Trusted Identity Propagation (TIP) token exchange flow for Amazon Q Business

def execute_tip_token_exchange(user_id, jwt_valid, bootstrap_role_ok, trusted_app):
    print(f"--- TRUSTED IDENTITY PROPAGATION AUDIT: {user_id} ---")

    # Step 1: Verify External IdP JWT
    if not jwt_valid:
        print("AUTH ERROR: Invalid JWT from External Identity Provider.")
        return
    print("Step 1 [IdP Auth]     : Signed JWT verified via Trusted Token Issuer.")

    # Step 2: OIDC CreateTokenWithIAM using Bootstrap Role
    if not bootstrap_role_ok:
        print("IAM ERROR: Bootstrap role failed sts:AssumeRoleWithWebIdentity.")
        return
    print("Step 2 [OIDC Exchange]: CreateTokenWithIAM returned bearer token (sts:identity-context).")

    # Step 3: STS AssumeRole with ProvidedContexts
    print("Step 3 [STS Context]  : sts:AssumeRole created Identity-Enhanced IAM Role Session.")

    # Step 4: Call Amazon Q Business ChatSync API
    print(f"Step 4 [Q ChatSync]   : Authorized call to {trusted_app} with user-scoped data filtering.")

# Run simulation for valid custom frontend request
execute_tip_token_exchange(
    user_id="alice@example.com",
    jwt_valid=True,
    bootstrap_role_ok=True,
    trusted_app="MyQBusinessApp"
)

--- TRUSTED IDENTITY PROPAGATION AUDIT: alice@example.com ---
Step 1 [IdP Auth]     : Signed JWT verified via Trusted Token Issuer.
Step 2 [OIDC Exchange]: CreateTokenWithIAM returned bearer token (sts:identity-context).
Step 3 [STS Context]  : sts:AssumeRole created Identity-Enhanced IAM Role Session.
Step 4 [Q ChatSync]   : Authorized call to MyQBusinessApp with user-scoped data filtering.


In [11]:
# Validate OAuth 2.0 Authorization Server configs and JWT claims for Trusted Token Issuer

OAUTH_SERVERS = {
    "Generic (Amazon Cognito)": {
        "scopes": "openid",
        "allowed_in_aws_event": True,
        "id_mapping": "email"
    },
    "Okta": {
        "scopes": "openid email offline_access",
        "allowed_in_aws_event": False,
        "id_mapping": "email or External ID (sub)"
    },
    "Microsoft Entra ID": {
        "scopes": "openid",
        "allowed_in_aws_event": False,
        "id_mapping": "email or External ID (sub/oid)"
    },
    "Auth0": {
        "scopes": "openid email offline_access",
        "allowed_in_aws_event": False,
        "id_mapping": "email or External ID (sub)"
    }
}

def validate_trusted_issuer_setup(provider, is_aws_event, jwt_payload):
    config = OAUTH_SERVERS.get(provider)
    print(f"=== TRUSTED TOKEN ISSUER AUDIT: {provider} ===")

    if is_aws_event and not config["allowed_in_aws_event"]:
        print("POLICY WARNING: Use Generic (Cognito) server during official AWS events.")
    else:
        print("Environment Check : APPROVED")

    required_claims = ["iss", "sub", "aud", "exp", "email"]
    missing = [c for c in required_claims if c not in jwt_payload]

    if missing:
        print(f"JWT ERROR: Missing required claims {missing}")
    else:
        print(f"JWT Claims Valid  : Yes (aud={jwt_payload['aud']})")
        print(f"OIDC Scopes       : {config['scopes']}")
        print(f"Attribute Mapping : {config['id_mapping']}\n")

# Test valid Cognito JWT payload from ./generate_token.py
sample_jwt = {
    "iss": "https://amazonaws.com",
    "sub": "user-12345",
    "aud": "client-app-98765",
    "exp": 1795000000,
    "email": "alice@example.com"
}
validate_trusted_issuer_setup("Generic (Amazon Cognito)", is_aws_event=True, jwt_payload=sample_jwt)

=== TRUSTED TOKEN ISSUER AUDIT: Generic (Amazon Cognito) ===
Environment Check : APPROVED
JWT Claims Valid  : Yes (aud=client-app-98765)
OIDC Scopes       : openid
Attribute Mapping : email



In [12]:
# Validate Customer Managed Application setup for Trusted Identity Propagation

def audit_customer_managed_app(app_arn, visibility, assignments_required, aud_claim, trusted_service, principal_arn):
    print("=== CUSTOMER MANAGED APPLICATION AUDIT ===")

    # Validate visibility and assignment settings for custom frontend
    if visibility == "Not visible" and not assignments_required:
        print("Portal Settings  : PASS (Hidden from portal; access controlled by Amazon Q)")
    else:
        print("Portal Settings  : WARNING (Check visibility/assignment rules)")

    print(f"Audience Claim   : {aud_claim}")
    print(f"Allowed Principal: {principal_arn} (Authorized for CreateTokenWithIAM)")
    print(f"Trusted Service  : {trusted_service} (Status: Configured access)")
    print(f"Exported Env Var : IDC_APPLICATION_ARN={app_arn}")

# Test Lab 5 Step 2 configuration
audit_customer_managed_app(
    app_arn="arn:aws:sso::123456789012:application/ssoins-example/apl-example",
    visibility="Not visible",
    assignments_required=False,
    aud_claim="client-app-98765",
    trusted_service="Amazon Q",
    principal_arn="arn:aws:iam::123456789012:root"
)

=== CUSTOMER MANAGED APPLICATION AUDIT ===
Portal Settings  : PASS (Hidden from portal; access controlled by Amazon Q)
Audience Claim   : client-app-98765
Allowed Principal: arn:aws:iam::123456789012:root (Authorized for CreateTokenWithIAM)
Trusted Service  : Amazon Q (Status: Configured access)
Exported Env Var : IDC_APPLICATION_ARN=arn:aws:sso::123456789012:application/ssoins-example/apl-example


In [13]:
# Simulate the 6-point diagnostic tool (diag_tool.py) for Trusted Identity Propagation

def run_diagnostic_tool(jwt_token, idc_app_arn, well_known_ok, app_auth_ok, trusted_q_ok, aud_match_ok, attr_map_ok):
    print(f"Running diagnostic check for ARN: {idc_app_arn}\n")

    if not jwt_token or not idc_app_arn:
        print("CRITICAL ERROR: Missing $JWT or $IDC_APPLICATION_ARN environment variables.")
        return

    checks = [
        ("1. OpenID Well-Known Configuration URL", well_known_ok),
        ("2. Customer Managed App Base Settings", True),
        ("3. Application Authorization Principal", app_auth_ok),
        ("4. Trusted AWS Managed App (Amazon Q)", trusted_q_ok),
        ("5. Matching Trusted Issuer & Aud Scope", aud_match_ok),
        ("6. Identity Store User Attribute Mapping", attr_map_ok)
    ]

    all_passed = True
    for check_name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {check_name}")
        if not passed:
            all_passed = False

    print("\n" + ("SUCCESS: All 6 checks passed! Ready for custom frontend." if all_passed else "ACTION REQUIRED: Resolve failed checks before proceeding."))

# Execute diagnostic check with valid Lab 5 outputs
run_diagnostic_tool(
    jwt_token="eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.sample",
    idc_app_arn="arn:aws:sso::123456789012:application/ssoins-example/apl-example",
    well_known_ok=True,
    app_auth_ok=True,
    trusted_q_ok=True,
    aud_match_ok=True,
    attr_map_ok=True
)

Running diagnostic check for ARN: arn:aws:sso::123456789012:application/ssoins-example/apl-example

[PASS] 1. OpenID Well-Known Configuration URL
[PASS] 2. Customer Managed App Base Settings
[PASS] 3. Application Authorization Principal
[PASS] 4. Trusted AWS Managed App (Amazon Q)
[PASS] 5. Matching Trusted Issuer & Aud Scope
[PASS] 6. Identity Store User Attribute Mapping

SUCCESS: All 6 checks passed! Ready for custom frontend.


In [14]:
# Validate custom AngularJS frontend config and 1-minute pre-expiry token refresh logic

REQUIRED_ENV_KEYS = [
    "oidcAuthorityUrl",
    "oidcClientId",
    "oidcScopes",
    "amazonQApplicationId",
    "bootstrapRoleArn",
    "identityRoleArn",
    "tokenExchangeApplicationArn"
]

def validate_frontend_and_refresh(env_config, token_ttl_seconds):
    print("=== CUSTOM FRONTEND & TOKEN REFRESH AUDIT ===")

    # 1. Verify all 7 required parameters in environment.vscode.ts
    missing_keys = [k for k in REQUIRED_ENV_KEYS if not env_config.get(k)]
    if missing_keys:
        print(f"CONFIG ERROR: Missing keys in environment.vscode.ts -> {missing_keys}")
        return
    print("Frontend Config  : PASS (All 7 environment variables verified)")
    print("Mutual Consent   : PASS (BootstrapRoleArn bound to Customer Managed App)")

    # 2. Verify 1-minute (60s) pre-expiry auto-refresh timer logic
    refresh_trigger_at = max(0, token_ttl_seconds - 60)
    print(f"Token Exchange   : One-time CreateTokenWithIAM enforced (No credential caching)")
    print(f"Auto-Refresh Rule: Timer triggers at {refresh_trigger_at}s (60s before {token_ttl_seconds}s expiry)")

# Test Lab 6 frontend configuration with 15-minute (900s) temporary credential TTL
sample_env = {
    "oidcAuthorityUrl": "https://amazonaws.com",
    "oidcClientId": "client-app-98765",
    "oidcScopes": "openid",
    "amazonQApplicationId": "q-app-id-12345",
    "bootstrapRoleArn": "arn:aws:iam::123456789012:role/AssetsStack-BootstrapRole",
    "identityRoleArn": "arn:aws:iam::123456789012:role/AssetsStack-IdentityRole",
    "tokenExchangeApplicationArn": "arn:aws:sso::123456789012:application/ssoins-example/apl-example"
}
validate_frontend_and_refresh(sample_env, token_ttl_seconds=900)

=== CUSTOM FRONTEND & TOKEN REFRESH AUDIT ===
Frontend Config  : PASS (All 7 environment variables verified)
Mutual Consent   : PASS (BootstrapRoleArn bound to Customer Managed App)
Token Exchange   : One-time CreateTokenWithIAM enforced (No credential caching)
Auto-Refresh Rule: Timer triggers at 840s (60s before 900s expiry)


In [15]:
# Final workshop completion summary and sandbox resource cleanup checklist

WORKSHOP_SUMMARY = {
    "Project": "IAM Identity Center: Empowering Secure Access to Generative AI Applications",
    "Level": "300 (Advanced Cloud Security)",
    "Core_Services": ["AWS IAM Identity Center", "Amazon Q Developer", "Amazon Q Business", "AWS STS"],
    "Security_Protocols": ["SAML 2.0", "SCIM v2.0", "OAuth 2.0 / OIDC", "Trusted Identity Propagation"]
}

CLEANUP_CHECKLIST = [
    "1. Delete CDK AssetsStack (Bootstrap & Identity Roles) via 'cdk destroy'",
    "2. Remove Customer Managed Application & Trusted Token Issuer from Identity Center",
    "3. Delete Amazon Q Business Application & Q Developer Subscriptions",
    "4. Delete base CloudFormation Stack (ws-template.yaml) to prevent ongoing AWS charges"
]

def generate_final_report():
    print("==========================================================")
    print("   WORKSHOP COMPLETED: END-TO-END SECURITY VERIFIED")
    print("==========================================================")
    for key, value in WORKSHOP_SUMMARY.items():
        print(f"{key:<20}: {value}")

    print("\n--- SANDBOX CLEANUP CHECKLIST (COST & SECURITY HYGIENE) ---")
    for item in CLEANUP_CHECKLIST:
        print(f"[ ] {item}")

generate_final_report()

   WORKSHOP COMPLETED: END-TO-END SECURITY VERIFIED
Project             : IAM Identity Center: Empowering Secure Access to Generative AI Applications
Level               : 300 (Advanced Cloud Security)
Core_Services       : ['AWS IAM Identity Center', 'Amazon Q Developer', 'Amazon Q Business', 'AWS STS']
Security_Protocols  : ['SAML 2.0', 'SCIM v2.0', 'OAuth 2.0 / OIDC', 'Trusted Identity Propagation']

--- SANDBOX CLEANUP CHECKLIST (COST & SECURITY HYGIENE) ---
[ ] 1. Delete CDK AssetsStack (Bootstrap & Identity Roles) via 'cdk destroy'
[ ] 2. Remove Customer Managed Application & Trusted Token Issuer from Identity Center
[ ] 3. Delete Amazon Q Business Application & Q Developer Subscriptions
[ ] 4. Delete base CloudFormation Stack (ws-template.yaml) to prevent ongoing AWS charges


In [16]:
# Enforce strict bottom-up AWS resource deletion order to prevent DependencyViolation errors

REQUIRED_DELETION_ORDER = [
    "Amazon Q Business Application",
    "Amazon Q Developer Subscriptions",
    "CDK AssetsStack (cdk destroy)",
    "IAM Identity Center Account Instance",
    "Base CloudFormation Stack (ws-template.yaml)"
]

def validate_cleanup_sequence(planned_steps):
    print("=== AWS SANDBOX CLEANUP SEQUENCE VALIDATOR ===")
    if planned_steps == REQUIRED_DELETION_ORDER:
        print("STATUS: SAFE TO EXECUTE - Bottom-up dependency order is 100% compliant.")
        for idx, step in enumerate(planned_steps, 1):
            print(f"Step {idx}: Delete {step}")
    else:
        print("DEPENDENCY ERROR: Incorrect deletion order! Follow strict bottom-up sequence:")
        for idx, step in enumerate(REQUIRED_DELETION_ORDER, 1):
            print(f"  Expected Step {idx}: {step}")

# Validate exact sequence from Lab 15 notes
validate_cleanup_sequence(REQUIRED_DELETION_ORDER)

=== AWS SANDBOX CLEANUP SEQUENCE VALIDATOR ===
STATUS: SAFE TO EXECUTE - Bottom-up dependency order is 100% compliant.
Step 1: Delete Amazon Q Business Application
Step 2: Delete Amazon Q Developer Subscriptions
Step 3: Delete CDK AssetsStack (cdk destroy)
Step 4: Delete IAM Identity Center Account Instance
Step 5: Delete Base CloudFormation Stack (ws-template.yaml)
